# cAPTure: graph OOF score-saturation audit

This notebook investigates why a model can have near-perfect OOF ranking while producing no alerts at every frozen operational budget. It is bound to the immutable four-model comparison and reads the same checksum-verified development OOF predictions.

The audit does not train, rescore, recalibrate, or alter thresholds. It does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Prepare the Colab environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_score_saturation import (
    run_graph_score_saturation_audit,
    validate_graph_score_saturation_audit,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("A GPU runtime is not required for this audit.")

## 2. Bind the immutable comparison and OOF artifacts

In [ ]:
BINDING_RUN_ID = "20260929T144234_934044Z_graph_training_binding"
COMPARISON_RUN_ID = "20261001T153351_608797Z_graph_four_model_comparison"
TRAINING_ROOT = DRIVE_ROOT / "graph_training_runs" / BINDING_RUN_ID
COMPARISON_DIR = DRIVE_ROOT / "graph_comparison_runs" / COMPARISON_RUN_ID

for path in (TRAINING_ROOT, COMPARISON_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required audit input is missing: {path}")
comparison_report = json.loads(
    (COMPARISON_DIR / "comparison_report.json").read_text(encoding="utf-8")
)
threshold_rows = []
for model_name, model_report in comparison_report["models"].items():
    for budget_name, item in model_report["thresholds"].items():
        threshold_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": item["threshold"],
            "threshold_above_one": item["threshold"] > 1.0,
            "worst_fold_false_alert_windows_per_hour": item["worst_fold_false_alert_windows_per_hour"],
        })
display(pd.DataFrame(threshold_rows).set_index(["model", "budget"]))
print("Source comparison:", COMPARISON_DIR)

## 3. Run or verify the read-only audit

The audit verifies the source comparison, completion files, metrics, and OOF checksums before scanning score tails. Set `RUN_AUDIT=True` after reviewing the bound thresholds. A completed audit is immutable and can be reloaded by setting its run ID.

In [ ]:
AUDIT_RUN_ID = None  # Set this to reuse a completed audit.
RUN_AUDIT = False
if AUDIT_RUN_ID is None:
    AUDIT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_score_saturation_audit"
    )
OUTPUT_DIR = (
    DRIVE_ROOT / "graph_score_saturation_audit_runs" / AUDIT_RUN_ID
)
print("Saturation-audit run ID:", AUDIT_RUN_ID)
print("Audit output:", OUTPUT_DIR)

if OUTPUT_DIR.exists():
    audit_report = validate_graph_score_saturation_audit(
        comparison_dir=COMPARISON_DIR,
        output_dir=OUTPUT_DIR,
    )
elif RUN_AUDIT:
    audit_report = run_graph_score_saturation_audit(
        comparison_dir=COMPARISON_DIR,
        training_root=TRAINING_ROOT,
        output_dir=OUTPUT_DIR,
    )
else:
    audit_report = None
    print("Set RUN_AUDIT=True to start the read-only saturation audit.")

## 4. Review the model-level diagnosis

A confirmed blocker requires all three signals: a selected threshold above one, benign windows stored exactly at one whose smallest attainable positive rate exceeds the budget, and attack packets tied at the same stored score.

In [ ]:
if audit_report is None:
    raise RuntimeError("Run or load the audit before displaying results.")
model_summary = pd.DataFrame(audit_report["tables"]["model_summary"])
display(model_summary.set_index("model_display")[[
    "stored_score_type", "primary_threshold",
    "primary_threshold_above_one",
    "primary_worst_fold_false_alert_windows_per_hour",
    "minimum_positive_worst_fold_false_alert_windows_per_hour",
    "exact_one_benign_windows", "exact_one_attack_packets",
    "exact_one_attack_packet_fraction", "primary_diagnosis",
    "oof_rescoring_from_checkpoint_recommended",
]])

## 5. Review every frozen false-alert budget

In [ ]:
budget_diagnostics = pd.DataFrame(
    audit_report["tables"]["budget_diagnostics"]
)
display(budget_diagnostics.set_index(["model_display", "budget"])[[
    "target_false_alert_windows_per_hour", "selected_threshold",
    "selected_threshold_above_one",
    "reported_worst_fold_false_alert_windows_per_hour",
    "minimum_positive_worst_fold_false_alert_windows_per_hour",
    "top_score_block_within_budget",
    "shared_exact_one_benign_attack_tie", "diagnosis",
]])

## 6. Inspect packet and window score tails

The detailed tables retain scenario and fold boundaries. Focus on the exact-one counts, maximum-tie sizes, distinct-score fractions, and upper quantiles.

In [ ]:
packet_tails = pd.DataFrame(audit_report["tables"]["packet_score_tails"])
window_tails = pd.DataFrame(
    audit_report["tables"]["window_maximum_score_tails"]
)
packet_columns = [
    "packets", "distinct_scores", "distinct_score_fraction",
    "score_q99", "score_q999", "score_q9999", "score_q99999",
    "maximum_score", "exact_one_scores", "exact_one_score_fraction",
    "maximum_tie_scores",
]
window_columns = [
    "wall_clock_windows", "observed_windows", "empty_windows",
    "window_maximum_score_q99", "window_maximum_score_q999",
    "window_maximum_score_q9999", "window_maximum_score_q99999",
    "maximum_window_maximum_score", "exact_one_window_maxima",
    "maximum_tie_windows",
]
display(packet_tails.set_index(
    ["model_display", "fold", "scenario", "packet_class"]
)[packet_columns])
display(window_tails.set_index(
    ["model_display", "fold", "scenario", "window_class"]
)[window_columns])

## 7. Review the saved audit report

The audit preserves the original comparison as evidence. If rescoring is recommended, the next artifact must repeat only OOF inference from the saved final checkpoints and store raw logits or float64 sigmoid scores under a new versioned contract.

In [ ]:
display(Markdown((OUTPUT_DIR / "review.md").read_text(encoding="utf-8")))
print("Audit status:", audit_report["status"])
print("Saturated models:", audit_report["saturated_models"])
print("OOF rescoring recommended:", audit_report["oof_rescoring_recommended"])
print("Threshold policy changed:", audit_report["threshold_policy_changed"])
print("Model training performed:", audit_report["model_training_performed"])
print("Held-out scenarios accessed:", audit_report["held_out_scenarios_accessed"])
print("Saved audit:", OUTPUT_DIR / "score_saturation_audit.json")
print("Saved review:", OUTPUT_DIR / "review.md")